### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [13]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)


import subprocess
import httpx

from openai import AsyncOpenAI
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel

windows_host = subprocess.check_output(
    "ip route | awk '/default/ {print $3}'",
    shell=True,
    text=True
).strip()

ollama_url = f"http://{windows_host}:11434/v1"

print("Usando Ollama en:", ollama_url)

ollama_client = AsyncOpenAI(
    base_url=ollama_url,
    api_key="ollama",
    http_client=httpx.AsyncClient(
        timeout=120,
        trust_env=False
    )
)

ollama_model = OpenAIChatCompletionsModel(
    model="qwen2.5:7b",
    openai_client=ollama_client
)

Usando Ollama en: http://172.21.96.1:11434/v1


In [14]:
from accounts import Account

In [15]:
account = Account.get("Ed")
account

Account(name='ed', balance=9546.094, strategy='', holdings={'AMZN': 6}, transactions=[3 shares of AMZN at 64.128 each., 3 shares of AMZN at 87.174 each.], portfolio_value_time_series=[('2026-05-13 12:57:23', 9867.616), ('2026-05-13 12:57:26', 9834.616), ('2026-05-15 11:25:03', 10128.094), ('2026-05-15 11:25:03', 10002.094)])

In [26]:
account.buy_shares("AMZN", 5, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ed", "balance": 9159.322, "strategy": "", "holdings": {"AMZN": 14}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 64.128, "timestamp": "2026-05-13 12:57:23", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 87.174, "timestamp": "2026-05-15 11:25:03", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 77.154, "timestamp": "2026-05-15 11:26:11", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 5, "price": 31.062, "timestamp": "2026-05-15 11:43:36", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-05-13 12:57:23", 9867.616], ["2026-05-13 12:57:26", 9834.616], ["2026-05-15 11:25:03", 10128.094], ["2026-05-15 11:25:03", 10002.094], ["2026-05-15 11:26:11", 9863.632], ["2026-05-15 11:26:17", 9782.632], ["2026-05-15 11:43:36", 100

In [27]:
account.report()

'{"name": "ed", "balance": 9159.322, "strategy": "", "holdings": {"AMZN": 14}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 64.128, "timestamp": "2026-05-13 12:57:23", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 87.174, "timestamp": "2026-05-15 11:25:03", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 77.154, "timestamp": "2026-05-15 11:26:11", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 5, "price": 31.062, "timestamp": "2026-05-15 11:43:36", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-05-13 12:57:23", 9867.616], ["2026-05-13 12:57:26", 9834.616], ["2026-05-15 11:25:03", 10128.094], ["2026-05-15 11:25:03", 10002.094], ["2026-05-15 11:26:11", 9863.632], ["2026-05-15 11:26:17", 9782.632], ["2026-05-15 11:43:36", 10055.322], ["2026-05-15 11:43:

In [28]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 64.128,
  'timestamp': '2026-05-13 12:57:23',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 87.174,
  'timestamp': '2026-05-15 11:25:03',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 77.154,
  'timestamp': '2026-05-15 11:26:11',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 5,
  'price': 31.062,
  'timestamp': '2026-05-15 11:43:36',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [29]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [20]:
mcp_tools

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None),
 Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None),
 Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', '

In [21]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "qwen2.5:7b"

In [30]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=ollama_model, mcp_servers=[mcp_server])
    #with trace("account_manager"):
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Your current balance is $9159.32.

### Now let's build our own MCP Client

In [23]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None), Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None), Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', 'ty

In [25]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

agent = Agent(name="account_manager", instructions=instructions, model=ollama_model, tools=openai_tools)
result = await Runner.run(agent, request)
display(Markdown(result.final_output))


Your current balance is $9314.632.

In [31]:
context = await read_accounts_resource("ed")
print(context)

{"name": "ed", "balance": 9159.322, "strategy": "", "holdings": {"AMZN": 14}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 64.128, "timestamp": "2026-05-13 12:57:23", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 87.174, "timestamp": "2026-05-15 11:25:03", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 77.154, "timestamp": "2026-05-15 11:26:11", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 5, "price": 31.062, "timestamp": "2026-05-15 11:43:36", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-05-13 12:57:23", 9867.616], ["2026-05-13 12:57:26", 9834.616], ["2026-05-15 11:25:03", 10128.094], ["2026-05-15 11:25:03", 10002.094], ["2026-05-15 11:26:11", 9863.632], ["2026-05-15 11:26:17", 9782.632], ["2026-05-15 11:43:36", 10055.322], ["2026-05-15 11:43:3

In [32]:
from accounts import Account
Account.get("ed").report()

'{"name": "ed", "balance": 9159.322, "strategy": "", "holdings": {"AMZN": 14}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 64.128, "timestamp": "2026-05-13 12:57:23", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 87.174, "timestamp": "2026-05-15 11:25:03", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 77.154, "timestamp": "2026-05-15 11:26:11", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 5, "price": 31.062, "timestamp": "2026-05-15 11:43:36", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-05-13 12:57:23", 9867.616], ["2026-05-13 12:57:26", 9834.616], ["2026-05-15 11:25:03", 10128.094], ["2026-05-15 11:25:03", 10002.094], ["2026-05-15 11:26:11", 9863.632], ["2026-05-15 11:26:17", 9782.632], ["2026-05-15 11:43:36", 10055.322], ["2026-05-15 11:43:

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>